# Iceberg Property Reviews — Explorer Notebook

**Uses**: `pyspark.sql` → Iceberg catalog → `property_reviews` table  
**Catalog**: Hadoop FileSystem (`./warehouse`)  
**Table**: `local.property_db.property_reviews`  
**Partition**: `country_code` × `review_year`

## 0 · Setup — SparkSession with Iceberg

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Adjust WAREHOUSE to your actual path if running from a different directory
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_DIR  = os.path.dirname(NOTEBOOK_DIR)   # one level up from notebooks/
WAREHOUSE    = os.path.join(PROJECT_DIR, 'warehouse')

CATALOG    = 'local'
FULL_TABLE = 'local.property_db.property_reviews'

spark = (
    SparkSession.builder
    .appName('IcebergViewer')
    .master('local[*]')
    .config(
        'spark.sql.extensions',
        'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions'
    )
    .config(f'spark.sql.catalog.{CATALOG}',
            'org.apache.iceberg.spark.SparkCatalog')
    .config(f'spark.sql.catalog.{CATALOG}.type', 'hadoop')
    .config(f'spark.sql.catalog.{CATALOG}.warehouse', WAREHOUSE)
    .config(
        'spark.jars.packages',
        'org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.4.2'
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('✅ SparkSession ready — Iceberg catalog:', CATALOG)
print('   Warehouse:', WAREHOUSE)

26/03/16 11:23:44 WARN Utils: Your hostname, w3e39 resolves to a loopback address: 127.0.1.1; using 192.168.0.227 instead (on interface enp2s0)
26/03/16 11:23:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/w3e39/Documents/spark-data-processing-iceberg/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/w3e39/.ivy2/cache
The jars for the packages stored in: /home/w3e39/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.4_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9765d17a-75fa-45ca-926e-34fa72a4f823;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.4_2.12;1.4.2 in central
:: resolution report :: resolve 139ms :: artifacts dl 8ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.4_2.12;1.4.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spar

✅ SparkSession ready — Iceberg catalog: local
   Warehouse: /home/w3e39/Documents/spark-data-processing-iceberg/warehouse


## 1 · Schema & Row Count

In [2]:
# Load the entire Iceberg table
df = spark.table(FULL_TABLE)

print(f'Total rows : {df.count()}')
print(f'Columns    : {len(df.columns)}')
print('\nSchema:')
df.printSchema()

26/03/16 11:23:58 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Total rows : 418
Columns    : 22

Schema:
root
 |-- property_id: long (nullable = true)
 |-- gen_id: string (nullable = true)
 |-- property_name: string (nullable = true)
 |-- property_slug: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- star_rating: double (nullable = true)
 |-- review_score: double (nullable = true)
 |-- published: boolean (nullable = true)
 |-- data_quality_flag: string (nullable = true)
 |-- review_id: long (nullable = true)
 |-- review_date: string (nullable = true)
 |-- review_individual_score: double (nullable = true)
 |-- review_language: string (nullable = true)
 |-- review_summary: string (nullable = true)
 |-- review_positive: string (nullable = true)
 |-- review_negative: string (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- reviewer_country: string (nullable = true)
 |-- reviewer_travel_purpose: string (nullable = true)
 |-- reviewer_type: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- 

## 2 · Sample Rows

In [3]:
# Register as a temp view so we can use plain SQL
df.createOrReplaceTempView('property_reviews')

spark.sql("""
    SELECT property_id, property_name, country_code,
           star_rating, review_score, review_year,
           reviewer_name, review_individual_score, data_quality_flag
    FROM   property_reviews
    LIMIT  10
""").show(truncate=False)

+-----------+-----------------------------------+------------+-----------+------------+-----------+-------------+-----------------------+-----------------+
|property_id|property_name                      |country_code|star_rating|review_score|review_year|reviewer_name|review_individual_score|data_quality_flag|
+-----------+-----------------------------------+------------+-----------+------------+-----------+-------------+-----------------------+-----------------+
|8635343    |Cedar Cottage-2592380              |GB          |3.0        |6.5         |2026       |Liu          |10.0                   |GOOD             |
|8899856    |Lee Valley-2598817                 |GB          |0.0        |7.4         |2026       |Steven       |1.0                    |NEEDS_REVIEW     |
|8899856    |Lee Valley-2598817                 |GB          |0.0        |7.4         |2026       |Steven       |1.0                    |NEEDS_REVIEW     |
|8899856    |Lee Valley-2598817                 |GB          |0.

## 3 · Reviews per Country (partition stats)

In [4]:
spark.sql("""
    SELECT  country_code,
            COUNT(DISTINCT property_id)    AS properties,
            COUNT(review_id)               AS total_reviews,
            ROUND(AVG(review_individual_score), 2) AS avg_review_score,
            ROUND(AVG(star_rating), 2)     AS avg_stars
    FROM    property_reviews
    GROUP BY country_code
    ORDER BY total_reviews DESC
""").show(40, truncate=False)

[Stage 4:===============================================>         (25 + 4) / 30]

+------------+----------+-------------+----------------+---------+
|country_code|properties|total_reviews|avg_review_score|avg_stars|
+------------+----------+-------------+----------------+---------+
|GB          |13        |78           |8.36            |2.51     |
|DE          |18        |74           |7.95            |3.3      |
|ID          |3         |30           |7.27            |2.33     |
|FR          |8         |22           |7.45            |2.86     |
|HR          |6         |18           |9.0             |3.75     |
|US          |7         |12           |7.67            |3.55     |
|JP          |2         |12           |8.67            |3.0      |
|FI          |1         |10           |9.8             |3.0      |
|MX          |3         |10           |8.4             |2.57     |
|AU          |3         |10           |8.8             |3.43     |
|ES          |2         |10           |9.8             |4.0      |
|BE          |1         |10           |9.6             |4.0   

## 4 · Review Volume by Year (second partition)

In [5]:
spark.sql("""
    SELECT  review_year,
            COUNT(*)                         AS total_reviews,
            ROUND(AVG(review_individual_score), 2) AS avg_score,
            COUNT(DISTINCT country_code)     AS countries_active
    FROM    property_reviews
    WHERE   review_year IS NOT NULL
    GROUP BY review_year
    ORDER BY review_year
""").show()

+-----------+-------------+---------+----------------+
|review_year|total_reviews|avg_score|countries_active|
+-----------+-------------+---------+----------------+
|       2023|           32|      8.0|               4|
|       2024|           58|     8.38|               8|
|       2025|          112|     8.09|              12|
|       2026|          118|     8.32|              15|
+-----------+-------------+---------+----------------+



## 5 · Top 10 Properties by Average Review Score

In [6]:
spark.sql("""
    SELECT  property_id,
            FIRST(property_name)                       AS name,
            FIRST(country_code)                        AS country,
            FIRST(star_rating)                         AS stars,
            ROUND(AVG(review_individual_score), 2)     AS avg_score,
            COUNT(review_id)                           AS num_reviews
    FROM    property_reviews
    WHERE   review_id IS NOT NULL
    GROUP BY property_id
    HAVING  num_reviews >= 2
    ORDER BY avg_score DESC
    LIMIT   10
""").show(truncate=False)

[Stage 16:==============================>                         (11 + 4) / 20]

+-----------+-----------------------------------+-------+-----+---------+-----------+
|property_id|name                               |country|stars|avg_score|num_reviews|
+-----------+-----------------------------------+-------+-----+---------+-----------+
|3138009    |Ash Cottage-2588974                |GB     |5.0  |10.0     |6          |
|7013181    |Cotswold View 2-2595780            |GB     |4.0  |10.0     |4          |
|9184791    |Mosscarr Barn-2598194              |GB     |4.0  |10.0     |8          |
|11589544   |Pinoso Mountain View Resort-393418 |ES     |4.0  |10.0     |6          |
|12071252   |Villa Acquerello-118134            |IT     |4.0  |10.0     |2          |
|15716133   |ITOSHI Kyoto Inn 二条城-235402     |JP     |3.0  |10.0     |2          |
|15814192   |Sky Luxury Apartment Shkoder-108175|AL     |0.0  |10.0     |6          |
|174022     |Bridget Inn-1378137                |FI     |3.0  |9.8      |10         |
|8719233    |Lavanda 2-96492                    |HR     |

## 6 · Data Quality Distribution

In [7]:
spark.sql("""
    SELECT  data_quality_flag,
            COUNT(DISTINCT property_id) AS properties,
            COUNT(*)                    AS rows
    FROM    property_reviews
    GROUP BY data_quality_flag
""").show()

[Stage 19:==============================================>         (25 + 4) / 30]

+-----------------+----------+----+
|data_quality_flag|properties|rows|
+-----------------+----------+----+
|     NEEDS_REVIEW|        51| 146|
|             GOOD|        48| 272|
+-----------------+----------+----+



## 7 · Reviewer Travel Purpose Breakdown

In [8]:
spark.sql("""
    SELECT  reviewer_travel_purpose,
            COUNT(*)                            AS reviews,
            ROUND(AVG(review_individual_score),2) AS avg_score
    FROM    property_reviews
    WHERE   reviewer_travel_purpose IS NOT NULL
    GROUP BY reviewer_travel_purpose
    ORDER BY reviews DESC
""").show()

+-----------------------+-------+---------+
|reviewer_travel_purpose|reviews|avg_score|
+-----------------------+-------+---------+
|                leisure|    256|      8.5|
|               business|     64|     7.09|
+-----------------------+-------+---------+



## 8 · Iceberg Metadata — Snapshots & Partitions

In [9]:
print('=== Snapshots ===')
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM   local.property_db.property_reviews.snapshots
""").show(truncate=False)

print('=== Partition summary ===')
spark.sql("""
    SELECT partition, record_count, file_count
    FROM   local.property_db.property_reviews.partitions
    ORDER BY partition
""").show(50, truncate=False)

=== Snapshots ===
+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|577620012072534400 |2026-03-16 11:11:35.717|append   |
|7549134412538562729|2026-03-16 11:14:49.823|append   |
+-------------------+-----------------------+---------+

=== Partition summary ===
+----------+------------+----------+
|partition |record_count|file_count|
+----------+------------+----------+
|{AL, NULL}|2           |2         |
|{AL, 2026}|6           |2         |
|{AT, 2025}|4           |2         |
|{AT, 2026}|2           |2         |
|{AU, NULL}|4           |2         |
|{AU, 2026}|10          |2         |
|{BE, 2026}|10          |2         |
|{BR, NULL}|14          |2         |
|{BR, 2024}|2           |2         |
|{CN, NULL}|2           |2         |
|{CO, NULL}|2           |2         |
|{DE, NULL}|12          |2         |
|{DE, 2023}|14          |2         |
|{DE, 2024}|20          |2 

## 9 · Partition Pruning Demo (fast scan)

In [10]:
# Only scans country_code='gb' AND review_year=2024 partitions
spark.sql("""
    SELECT  property_name, reviewer_name,
            review_date, review_individual_score, review_summary
    FROM    property_reviews
    WHERE   country_code = 'GB'
      AND   review_year  = 2024
    ORDER BY review_individual_score DESC
""").show(truncate=False)

+------------------------+-------------+-----------+-----------------------+--------------------------------------------------------------------------------------------------+
|property_name           |reviewer_name|review_date|review_individual_score|review_summary                                                                                    |
+------------------------+-------------+-----------+-----------------------+--------------------------------------------------------------------------------------------------+
|Anvil Barn-2588970      |Thomas       |2024-08-16 |10.0                   |A lovely holiday                                                                                  |
|Mosscarr Barn-2598194   |Natalie      |2024-06-16 |10.0                   |NULL                                                                                              |
|Mosscarr Barn-2598194   |Louise       |2024-11-04 |10.0                   |NULL                                        

## 10 · Reviewer Type vs Star Rating Correlation

In [11]:
spark.sql("""
    SELECT  reviewer_type,
            ROUND(AVG(star_rating), 2)             AS avg_stars,
            ROUND(AVG(review_individual_score), 2) AS avg_review_score,
            COUNT(*)                               AS total_reviews
    FROM    property_reviews
    WHERE   reviewer_type IS NOT NULL
    GROUP BY reviewer_type
    ORDER BY avg_review_score DESC
""").show(truncate=False)

+--------------------+---------+----------------+-------------+
|reviewer_type       |avg_stars|avg_review_score|total_reviews|
+--------------------+---------+----------------+-------------+
|couple              |2.7      |8.67            |114          |
|family_with_children|3.33     |8.47            |86           |
|extended_group      |2.57     |7.71            |42           |
|solo_traveller      |3.18     |7.56            |78           |
+--------------------+---------+----------------+-------------+



## 11 · Matplotlib Charts (optional)

In [12]:
try:
    import matplotlib.pyplot as plt
    import pandas as pd

    # ── Chart 1: Reviews per country ─────────────────────────────────────────
    country_pd = spark.sql("""
        SELECT country_code, COUNT(review_id) AS reviews
        FROM   property_reviews
        WHERE  review_id IS NOT NULL
        GROUP BY country_code
        ORDER BY reviews DESC
        LIMIT 15
    """).toPandas()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(country_pd['country_code'], country_pd['reviews'], color='steelblue')
    axes[0].set_title('Reviews per Country (top 15)')
    axes[0].set_xlabel('Country Code')
    axes[0].set_ylabel('Number of Reviews')
    axes[0].tick_params(axis='x', rotation=45)

    # ── Chart 2: Avg score per year ───────────────────────────────────────────
    year_pd = spark.sql("""
        SELECT  review_year,
                ROUND(AVG(review_individual_score), 2) AS avg_score
        FROM    property_reviews
        WHERE   review_year IS NOT NULL
        GROUP BY review_year
        ORDER BY review_year
    """).toPandas()

    axes[1].plot(year_pd['review_year'], year_pd['avg_score'],
                 marker='o', linewidth=2, color='darkorange')
    axes[1].set_title('Average Review Score by Year')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Avg Score')
    axes[1].set_ylim(0, 10)
    axes[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, 'notebooks', 'iceberg_charts.png'),
                dpi=120, bbox_inches='tight')
    plt.show()
    print('Charts saved → notebooks/iceberg_charts.png')

except ImportError:
    print('matplotlib not installed — skipping charts (pip install matplotlib pandas)')

matplotlib not installed — skipping charts (pip install matplotlib pandas)


## 12 · Clean Up

In [ ]:
spark.stop()
print('SparkSession stopped.')